# Breast Cancer Detection

You're an ML engineer at a health technology company. Doctors take measurements from breast tissue samples (radius, texture, smoothness, and others) and they need to know whether a tumour is **malignant** or **benign**. Your job is to build a classifier that can help with that decision.

The stakes are high. A missed diagnosis could cost a life.

This is a **binary classification** problem: we're predicting one of two categories. The model will look at a set of measurements and output a prediction, malignant or benign.

## Loading and exploring the data

Before building any model, we need to understand the data. What features do we have? How many samples? Are there missing values? For a classification problem, the balance between classes matters too.

Let's start by loading the dataset and taking a look.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

In [ ]:
df = pd.read_csv("./breast_cancer.csv")
df.head()

Each row is one tumour sample. The columns are physical measurements taken from the tissue, like radius, texture, perimeter, and smoothness. The last column, `diagnosis`, is what we're trying to predict.

Let's check the size of the dataset and whether there are any missing values we'd need to deal with.

In [ ]:
print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")

No missing values, so we can work with the data as-is. Now let's check something important for classification: how are the two classes distributed? If one class heavily outnumbers the other, it can mislead the model (and us).

In [ ]:
print(df["diagnosis"].value_counts())
print(f"\nBenign: {(df['diagnosis'] == 'benign').mean():.1%}")
print(f"Malignant: {(df['diagnosis'] == 'malignant').mean():.1%}")

About 63% benign, 37% malignant. Not drastically imbalanced, but enough that a lazy model could game the numbers.

## Train/test split

Time to split the data. This time we add `stratify=y`, which ensures both the training and test sets keep the same ratio of malignant to benign. That's important for classification, especially when classes aren't perfectly balanced.

In [ ]:
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {X_train.shape[0]} samples")
print(f"Test:     {X_test.shape[0]} samples")
print(f"\nTraining class balance:\n{y_train.value_counts(normalize=True).round(3)}")

## Feature scaling

The features are on very different scales. Area is in the hundreds, while smoothness is a small decimal. `StandardScaler` centres each feature to mean 0 and standard deviation 1. This helps logistic regression converge properly.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Training a logistic regression

Logistic regression outputs a probability (between 0 and 1) that a tumour is malignant. Under the hood, it's trained using **binary cross-entropy**, a loss function that penalises confident wrong predictions most harshly.

If the model says "95% benign" and the tumour is actually malignant, the penalty is severe. If it says "55% benign" and gets it wrong, the penalty is much smaller. This pushes the model to be well-calibrated in its confidence.

In [ ]:
model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
print("Model trained!")

## Accuracy is not enough

Our model produces predictions, but how do we know if they're good? The most intuitive metric is **accuracy**: the percentage of predictions the model got right. But accuracy can be misleading.

To see why, let's compare our model to the laziest possible baseline: a model that just predicts "benign" for every single sample, regardless of the measurements.

In [ ]:
# Generate predictions from our model
y_pred = model.predict(X_test_scaled)

# Our model's accuracy
model_accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {model_accuracy:.3f}")

# Naive baseline: predict "benign" for every sample
naive_accuracy = (y_test == "benign").mean()
print(f"Naive baseline accuracy (always predict benign): {naive_accuracy:.3f}")

# How many cancers does the naive baseline catch?
naive_recall = 0  # it never predicts malignant, so it catches nothing
print(f"\nNaive baseline recall: {naive_recall}")
print("The naive model misses every single cancer.")

The naive model gets roughly 63% accuracy by predicting "benign" every time. It catches zero cancers. Our model does much better, but accuracy alone doesn't tell us how many cancers we're missing.

## Introducing recall

In medical screening, the critical question is: **of all the actual cancers, how many did we catch?**

This is **recall** (also called sensitivity). A recall of 1.0 means we caught every cancer. A recall of 0.8 means we missed 20% of them.

Let's calculate it manually first.

In [ ]:
# Calculate recall manually
# Step 1: How many actual malignant cases are in y_test?
actual_malignant = (y_test == "malignant").sum()
print(f"Actual malignant cases: {actual_malignant}")

# Step 2: Of those, how many did the model correctly predict as malignant?
# We need cases where BOTH y_test == "malignant" AND y_pred == "malignant"
correctly_caught = ((y_test == "malignant") & (y_pred == "malignant")).sum()
print(f"Correctly caught: {correctly_caught}")

# Step 3: Recall = correctly caught / actual malignant
manual_recall = correctly_caught / actual_malignant
print(f"Manual recall: {manual_recall:.3f}")

Now let's verify our manual calculation using sklearn's built-in function. It's good practice to compute metrics by hand at least once so you understand what the number actually means, then use the library version for convenience going forward.

In [ ]:
recall = recall_score(y_test, y_pred, pos_label="malignant")
print(f"Recall (sklearn): {recall:.3f}")

## The cost of missing a cancer

There aren't neat right answers to these questions, but they're worth thinking through.

**1.** If the model misses 1 in 10 cancers, what does that mean in a hospital processing 1,000 biopsies a year?

**2.** A clinician using this tool: would they prefer higher recall (catch more cancers, but also flag more benign tumours for further testing) or higher accuracy?

**3.** Is there a recall threshold below which this tool would be unacceptable? What might that be?

## Adjusting the threshold

By default, logistic regression uses a 0.5 threshold: if the predicted probability of malignant is ≥ 0.5, it predicts malignant. But we can move this threshold.

Lowering it means we predict malignant more often, catching more cancers (higher recall) but also flagging more benign cases as suspicious.

In [ ]:
probabilities = model.predict_proba(X_test_scaled)
# The model outputs probabilities for [benign, malignant]
# We want the probability of malignant
malignant_index = list(model.classes_).index("malignant")
malignant_proba = probabilities[:, malignant_index]

thresholds = np.arange(0.1, 0.9, 0.05)
recalls = []

for threshold in thresholds:
    pred_labels = np.where(malignant_proba >= threshold, "malignant", "benign")
    recalls.append(recall_score(y_test, pred_labels, pos_label="malignant"))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, recalls, marker="o", markersize=4)
plt.xlabel("Decision threshold")
plt.ylabel("Recall (malignant)")
plt.title("How the decision threshold affects recall")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

As we lower the threshold, recall increases and we catch more cancers. But this comes at a cost: more benign tumours are flagged as suspicious, meaning more patients face unnecessary anxiety and follow-up procedures. In the next unit, you'll learn the name for this trade-off and how to measure it properly.